In [1]:
import pinocchio as pin
import pinocchio.casadi as cpin
import casadi as cs
from pinocchio.robot_wrapper import RobotWrapper
import sys, os
import numpy as np
from enum import Enum
sys.path.append(os.path.abspath("../../src"))
from symbolic_generator import SymbolicGenerator, KinematicsOrientation

symb_gen = SymbolicGenerator('pineapple_8dof.urdf', 
                             floating = True,
                             kinematics_bodies=['L_wheel', 'R_wheel'],
                             actuated_dofs = slice(6,14),
                             kinematics_ori = KinematicsOrientation.Quaternion,
                             gen_dir="./generated_code/pineapple_8dof_quat",
                             write_files = False)
symb_gen.generate()
model, data = symb_gen.model, symb_gen.data

Loaded robot model from: pineapple_8dof.urdf
----------- Model Details -----------
	Floating base = true
	# of configs = 15
	# of DoFs = 14

	Bodies: ['base_link', 'L_hip_joint', 'L_hip', 'L_thigh_joint', 'L_thigh', 'L_calf_joint', 'L_calf', 'L_wheel_joint', 'L_wheel', 'R_hip_joint', 'R_hip', 'R_thigh_joint', 'R_thigh', 'R_calf_joint', 'R_calf', 'R_wheel_joint', 'R_wheel', 'cam_joint', 'cam', 'imu_joint', 'imu_link']

	Joints: ['universe', 'root_joint', 'L_hip_joint', 'L_thigh_joint', 'L_calf_joint', 'L_wheel_joint', 'R_hip_joint', 'R_thigh_joint', 'R_calf_joint', 'R_wheel_joint']

	Kinematics: ['L_wheel', 'R_wheel']

	Orienation representation: Quaternion

	State vector order: ['x', 'y', 'z', 'q_w', 'q_x', 'q_y', 'q_z', 'L_hip_joint', 'L_thigh_joint', 'L_calf_joint', 'L_wheel_joint', 'R_hip_joint', 'R_thigh_joint', 'R_calf_joint', 'R_wheel_joint', 'lin_v_x', 'lin_v_y', 'lin_v_z', 'ang_v_x', 'ang_v_y', 'ang_v_z', 'L_hip_joint', 'L_thigh_joint', 'L_calf_joint', 'L_wheel_joint', 'R_hip_j

In [3]:
symb_gen.inverse_dynamics.generate("test.c", symb_gen.gen_opts)

'test.c'

In [31]:
v1 = cs.vec(symb_gen.dtau_dx)
v2 = cs.vec(symb_gen.dtau_dv_dot)
J1 = (cs.jacobian(v1, symb_gen.x))
J2 = (cs.jacobian(v1, symb_gen.v_dot))
J3 = (cs.jacobian(v2, symb_gen.x))
J4 = (cs.jacobian(v2, symb_gen.v_dot))
J = cs.vertcat(cs.horzcat(J1, J2), cs.horzcat(J3, J4))
test = cs.Function("test", [symb_gen.x, symb_gen.v_dot], [J,])
test.generate("test.c", symb_gen.gen_opts)

'test.c'

In [42]:
v1 = symb_gen.dtau_dx.T@symb_gen.tau
v2 = symb_gen.dtau_dv_dot.T@symb_gen.tau
J1 = (cs.jacobian(v1, symb_gen.x))
J2 = (cs.jacobian(v1, symb_gen.v_dot))
J3 = (cs.jacobian(v2, symb_gen.x))
J4 = (cs.jacobian(v2, symb_gen.v_dot))
J = cs.vertcat(cs.horzcat(J1, J2), cs.horzcat(J3, J4))
test = cs.Function("test", [symb_gen.x, symb_gen.v_dot, symb_gen.tau], [J,])
test.generate("test.c", symb_gen.gen_opts)
J.shape

(43, 43)

In [46]:
jtvp = cs.jtimes(symb_gen.tau_out, symb_gen.x, symb_gen.tau, True)
J = cs.jacobian(jtvp, symb_gen.x)
test = cs.Function("test", [symb_gen.x, symb_gen.v_dot, symb_gen.tau], [J,])
test.generate("test.c", symb_gen.gen_opts)
J.shape

(29, 29)

In [54]:
q_f = symb_gen.E.T@cs.jtimes(symb_gen.kinematics, symb_gen.q, symb_gen.force, True) 
cs.jacobian(q_f, symb_gen.x)

SX(sparse: 14-by-29, 112 nnz
 @1=(force_8+force_1),
 @2=2,
 @3=(force_9+force_2),
 @4=(force_7+force_0),
 @5=0.5,
 @6=(@2*x_6),
 @7=cos(x_11),
 @8=1e-12,
 @9=1,
 @10=(@2*x_5),
 @11=(@10*x_5),
 @12=(@6*x_6),
 @13=(@9-(@11+@12)),
 @14=cos(x_12),
 @15=(@6*x_4),
 @16=(@10*x_3),
 @17=(@15+@16),
 @18=(@10*x_4),
 @19=(@6*x_3),
 @20=(@18-@19),
 @21=sin(x_11),
 @22=((@17*@7)-(@20*@21)),
 @23=sin(x_12),
 @24=((@13*@14)-(@22*@23)),
 @25=cos(x_13),
 @26=((@13*@23)+(@22*@14)),
 @27=sin(x_13),
 @28=((@24*@25)-(@26*@27)),
 @29=cos(x_14),
 @30=((@24*@27)+(@26*@25)),
 @31=sin(x_14),
 @32=((@28*@29)-(@30*@31)),
 @33=(@2*x_4),
 @34=(@33*x_4),
 @35=(@9-(@34+@12)),
 @36=(@6*x_5),
 @37=(@33*x_3),
 @38=(@36-@37),
 @39=((@35*@7)+(@38*@21)),
 @40=(@15-@16),
 @41=(@9-(@34+@11)),
 @42=(@36+@37),
 @43=((@41*@7)-(@42*@21)),
 @44=((@40*@14)-(@43*@23)),
 @45=((@40*@23)+(@43*@14)),
 @46=((@44*@25)-(@45*@27)),
 @47=((@44*@27)+(@45*@25)),
 @48=((@46*@31)+(@47*@29)),
 @49=((@32+@39)+@48),
 @50=(@8<(@9+@49)),
 @51=sqrt((